# Notebook para la extraccioón de información de los documentos e indexacion 


In [2]:
import os
import fitz
from langchain_experimental.text_splitter import SemanticChunker
from dotenv import load_dotenv, find_dotenv
from openai import embeddings
from utils.ai_services import AzureServices
from services.indexing_service import DocumentProcessingPipeline
import pdb
import numpy as np
import json
import pandas as pd
from datetime import datetime
# Cargar variables de entorno (asegúrate que el .env tenga las de Azure)
load_dotenv(find_dotenv())

ModuleNotFoundError: No module named 'bs4'

## Lectura de de documentos

In [ ]:
db= os.getenv("AZURE_COSMOSDB_DATABASE_NAME")
collection= os.getenv("AZURE_COSMOSDB_COLLECTION_NAME")     
conection_tring= os.getenv("AZURE_COSMOSDB_ENDPOINT")

In [ ]:
# Inicializar servicios de Azure
blob_storage = AzureServices.AzureBlobStorage()
di = AzureServices.DocumentIntelligence()

doc= DocumentProcessingPipeline().reading_processing_documents()
# openai = AzureServices.AzureOpenAI()
# cosmos = AzureServices.CosmosDB(db_name=db, collection_names=collection, connection_string=conection_tring)

In [14]:
blobs= ["Bloque Metro/2020-02-12-javier-alonso-quintero-agudelo_primera.pdf"]
with open('kb_id_to_name.json', 'r', encoding='utf-8') as archivo:
    kb_ids= json.load(archivo)

In [ ]:
with open('Bloque Metro/2020-02-12-javier-alonso-quintero-agudelo_primera (1).pdf', 'rb') as f:
    resultado = di.extract_doc_text(file_obj=f.read(), batch_size=2000)

In [17]:
data_json = []

batch_size = 500
for blob in blobs:
    pdf_bytes = blob_storage.download_file(blob)
    print("Bytes descargados:", len(pdf_bytes) if pdf_bytes else "0")
    if pdf_bytes is None:
        print("No se pudo descargar el archivo.")
    
    kd_id = next((k for k, v in kb_ids.items() if v == blob.split("/")[0]), None)
    # Abrir el PDF solo para contar páginas
    pdf_doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    total_pages = pdf_doc.page_count
    pdf_doc.close()

    data_json = []

    # Procesar por batches
    for start in range(0, total_pages, batch_size):
        end = min(start + batch_size, total_pages)
        # Extraer el batch de páginas
        temp_pdf = fitz.open()
        for page_num in range(start, end):
            temp_pdf.insert_pdf(fitz.open(stream=pdf_bytes, filetype="pdf"), from_page=page_num, to_page=page_num)
        batch_bytes = temp_pdf.tobytes()
        temp_pdf.close()

        # Extraer texto/tablas del batch usando Azure DI
        texto, tablas, num_paginas_batch, poller = di.extract_doc_text(file_obj=batch_bytes, batch_size=batch_size)

        data_poller = poller.result()
        paragraphs = data_poller.paragraphs
        tables = data_poller.tables
        filtered_paragraphs = [p for p in paragraphs if p.role not in ['pageHeader', 'pageFooter', 'footnote']]

        # Ajuste: suma el offset del batch al número de página de cada párrafo/tabla
        for para in filtered_paragraphs:
            content = para.content
            if para.bounding_regions and len(para.bounding_regions) > 0:
                # Aquí está la clave:
                page_number = start + para.bounding_regions[0].page_number  # sumas el offset!
                polygon = [(point.x, point.y) for point in para.bounding_regions[0].polygon]
            else:
                page_number = None
                polygon = None
            data_json.append({
                "docnm_kwd": blob.split("/")[-1].replace(".pdf", ""),
                "docnm": blob.split("/")[-1],
                "bloque": blob.split("/")[0],
                "kb_id": kd_id,
                "content": content,
                "page_number": page_number
            })

        # Para las tablas igual:
        for i, table in enumerate(tables):
            page_number_table = start + table.bounding_regions[0].page_number
            data_json.append({
                "docnm_kwd": blob.split("/")[-1].replace(".pdf", ""),
                "docnm": blob.split("/")[-1],
                "bloque": blob.split("/")[0],
                "kb_id": kd_id,
                "content": tablas[i],
                "page_number": page_number_table
            })

Descargando archivo: Bloque Metro/2020-02-12-javier-alonso-quintero-agudelo_primera.pdf
Archivo descargado exitosamente: Bloque Metro/2020-02-12-javier-alonso-quintero-agudelo_primera.pdf
Bytes descargados: 30142872
Procesando páginas 0-499 | Tamaño del batch: 74.53 MB


c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().all().all():
c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().a

Procesando páginas 0-499 | Tamaño del batch: 80.22 MB
Procesando páginas 0-499 | Tamaño del batch: 67.83 MB


c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().all().all():
c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().a

Procesando páginas 0-499 | Tamaño del batch: 65.03 MB


c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().all().all():
c:\Users\JuanCamiloTorresSala\OneDrive - DATAKNOW S.A.S\Codigo para proyectos\Fiscalia\DK_sentencias_975_react\backend\app\utils\ai_services.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if tb.apply(lambda x: x.str.strip()).replace('', np.nan).isnull().a

Procesando páginas 0-277 | Tamaño del batch: 47.12 MB


In [ ]:
data_json, tables 

In [18]:
# Supongamos que data_json es tu diccionario de Python
with open("pdfs_data.json", "w", encoding="utf-8") as f:
    json.dump(data_json, f, indent=4, ensure_ascii=False)

In [ ]:
data_df = pd.DataFrame(data_json)

In [ ]:
data_df

In [ ]:
data_df.to_csv("C:/Users/JuanCamiloTorresSala/Downloads/data.csv", index=False, encoding='utf-8-sig')

# Sección de pruebas

In [ ]:
data_poller= poller.result()  # Esperar a que el procesamiento se complete

In [ ]:
data_poller.paragraphs

In [ ]:
len(filtered_paragraphs)



In [ ]:
data_ej= []
i= 0
for table in table_ej:
        # Sacar el numero de página donde inicia la tabla
        print("Tabla:", table)
        page_number_table = table.bounding_regions[0].page_number


        data_ej.append({
            "docnm_kwd": blob.split("/")[-1].replace(".pdf", ""),
            "bloque": blob.split("/")[0],
            "kb_id": kd_id,
            "content": tablas[i],
            "page_number": page_number_table
        })
        i+=1



In [ ]:
data_ej

In [ ]:
data = []
for para in filtered_paragraphs:
    # Extrae contenido
    content = para.content
    # Extrae la página y el polígono del primer bounding_region (si existe)
    if para.bounding_regions and len(para.bounding_regions) > 0:
        page_number = para.bounding_regions[0].page_number
        polygon = [(point.x, point.y) for point in para.bounding_regions[0].polygon]
    else:
        page_number = None
        polygon = None
    data.append({
        "content": content,
        "page_number": page_number,
        "polygon": polygon
    })
    


# Pruebas Grafo

In [ ]:
from langchain_core.messages import HumanMessage
from services.graph import graph
from utils.messages_serialize import serialize_message, deserialize_message
from services.ai_services import AzureServices
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

In [ ]:
db= os.getenv("AZURE_COSMOSDB_DATABASE_NAME")
collection= os.getenv("AZURE_COSMOSDB_COLLECTION_NAME")     
conection_tring= os.getenv("AZURE_COSMOSDB_ENDPOINT")

In [ ]:
# cosmos= AzureServices.CosmosDB(db_name=db, collection_names="Graphs_Users", connection_string=conection_tring)

In [ ]:
state = {
    "user_id": "usuario_test",
    "messages": [HumanMessage(content="¿Cuáles son los derechos de las víctimas según la ley 975?")],
    "response": ""
}



In [ ]:
response = await graph.ainvoke(state)
print("Bot:", response["response"])


In [ ]:
response

In [ ]:
state = {
    "user_id": response["user_id"],              # O simplemente "usuario_test"
    "messages": response["messages"] + [HumanMessage(content="¿Qué casos hay sobre violencia sexual?")],
    "response": ""
}
response = await graph.ainvoke(state)
print("Bot:", response["response"])

In [ ]:
response["messages"]

In [ ]:
# for m in response["messages"]:
#     serialize_m= serialize_message(m,"usuario_test")
#     cosmos.insert_message(serialize_m, "Graphs_Users")
#     # print(serialize_m)

# Pruebas blob

In [ ]:
base_url= "https://fgnadlsententranscripts.blob.core.windows.net/sentencias975/"
token= "?sp=racwdlmeop&st=2025-07-17T14:57:23Z&se=2026-08-01T23:12:23Z&sv=2024-11-04&sr=c&sig=%2BzLmylrFku3cr%2F8rXjvAGgdl1nrhEBOl5WRr4uiunxc%3D"
path_file= "Bloque Bananero/Sentencia-Hebert-Veloza-Garcia-42799-20-nov-2014.pdf"
link_file= f"{base_url}{path_file}{token}"
print(link_file)

In [ ]:
blob_storage=AzureServices().AzureBlobStorage()

In [ ]:
blobs = blob_storage.list_blobs()
print(len(blobs))

In [ ]:
ordenados= blob_storage.list_blobs_by_page_count()
# for nombre, paginas in ordenados:
#     print(f"{nombre} => {paginas} páginas")

In [ ]:
ordenados